In [0]:
# ============================================================
# PARAMÈTRES
# ============================================================
dbutils.widgets.text("catalog",        "banking")
dbutils.widgets.text("schema_bronze",  "bronze")
dbutils.widgets.text("schema_silver",  "silver")

catalog       = dbutils.widgets.get("catalog")
schema_bronze = dbutils.widgets.get("schema_bronze")
schema_silver = dbutils.widgets.get("schema_silver")

# Tables
source_table     = f"{catalog}.{schema_bronze}.bronze_branches"
target_silver    = f"{catalog}.{schema_silver}.silver_branches"
monitoring_table = f"{catalog}.{schema_bronze}.execution_monitoring"

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from pyspark.sql.functions import row_number


df = spark.table(source_table)
df=df.filter(F.col("branch_code").isNotNull())
df=df.withColumn('branch_name' ,F.trim(F.col('branch_name')))
df=df.withColumn('city' ,F.trim(F.col('city')))
df=df.withColumn('state' ,F.trim(F.col('state')))
df=df.withColumn('region' ,F.trim(F.col('region')))
df=df.filter(F.col("branch_name").isNotNull())
df=df = df.withColumn('silver_loaded_at', F.current_timestamp())
df.write.mode("overwrite").option("delta.feature.allowColumnDefaults", "supported").saveAsTable(target_silver)
print(f"Table {target_silver} created successfully")